# Wine Reviews — Advanced EDA (segment statistics)

Companion to `01_eda_basic`. Here we break **price** and **rating** down by
segment — country, variety, wine type, bottle age, and review-text keyword
cues — reporting count, mean and median for each group.

Loaded from the cleaned **Silver** parquet; the robust keyword features
(`features_keywords_robust.parquet`) are joined for section 12.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

SILVER_PATH = r"..\..\.data\wine_reviews_silver.parquet"
KW_ROBUST_PATH = r"..\..\features\features_keywords_robust.parquet"

df = pd.read_parquet(SILVER_PATH)

# Bottle age at review (year of review - vintage). NV wines have no vintage -> NaN.
review_year = pd.to_datetime(df["date_of_review"], errors="coerce").dt.year
df["age_at_review"] = review_year - df["vintage"]


def agg_pr(data, by, min_n=30):
    """Count + mean/median of retail & rating per group, kept where n >= min_n."""
    g = (
        data.dropna(subset=["retail", "rating"])
        .groupby(by, observed=True)
        .agg(
            n=("retail", "size"),
            retail_mean=("retail", "mean"),
            retail_median=("retail", "median"),
            rating_mean=("rating", "mean"),
            rating_median=("rating", "median"),
        )
    )
    return g[g["n"] >= min_n].sort_values("n", ascending=False).round(2)


print(f"Rows: {len(df):,}  |  age_at_review null: {df['age_at_review'].isna().sum():,}")

Rows: 135,192  |  age_at_review null: 4,618


## 8. Mean and median (price, rating) by country

In [2]:
by_country = agg_pr(df, "country", min_n=100)
display(by_country.head(20))

top = by_country.head(15).reset_index()
fig = px.bar(
    top, x="country", y="retail_median", color="rating_median",
    color_continuous_scale="Viridis",
    title="Median retail by country (top 15 by review count; colour = median rating)",
    labels={"retail_median": "Median retail (USD)", "rating_median": "Median rating"},
)
fig.update_layout(height=460)
fig.show()

,n,retail_mean,retail_median,rating_mean,rating_median
country,,,,,
USA,54625,47.56,40.0,90.74,91.0
France,26622,42.33,27.0,90.05,90.0
Italy,18242,48.55,32.0,90.50,90.0
Portugal,6456,38.00,20.0,89.37,89.0
Spain,5187,39.52,25.0,90.41,91.0
Australia,2608,51.70,29.0,90.51,90.0
Austria,2581,38.00,30.0,90.64,90.0
Argentina,2529,30.79,21.0,88.97,89.0
Germany,2077,37.29,27.0,89.73,90.0


## 9. Mean and median (price, rating) by variety

In [3]:
by_variety = agg_pr(df, "varietal_label", min_n=200)
display(by_variety.head(25))

top = by_variety.head(20).reset_index()
fig = px.scatter(
    top, x="rating_median", y="retail_median", size="n", text="varietal_label",
    title="Median retail vs median rating by variety (top 20 by count; size = n)",
    labels={"retail_median": "Median retail (USD)", "rating_median": "Median rating"},
)
fig.update_traces(textposition="top center")
fig.update_layout(height=560)
fig.show()

,n,retail_mean,retail_median,rating_mean,rating_median
varietal_label,,,,,
Pinot Noir,14779,56.54,50.0,91.33,92.0
Chardonnay,10556,45.81,38.0,90.75,91.0
Cabernet Sauvignon,8510,62.66,45.0,90.54,91.0
Red Blend,7048,42.55,32.0,90.10,90.0
Bordeaux-style Red Blend,6694,48.24,30.0,90.17,90.0
Sauvignon Blanc,5384,27.36,24.0,89.21,89.0
Rosé,4977,22.49,20.0,88.55,88.0
Riesling,3848,35.15,27.0,90.30,90.0
Syrah,3588,53.60,48.0,91.74,92.0


## 10. Mean and median (price, rating) by wine_type

In [4]:
by_type = agg_pr(df, "wine_type", min_n=30)
by_type

,n,retail_mean,retail_median,rating_mean,rating_median
wine_type,,,,,
Red,75503,50.16,39.0,90.67,91.0
White,37071,33.32,26.0,89.94,90.0
Sparkling,6602,52.98,40.0,90.28,90.0
Rose,6516,22.82,20.0,88.71,89.0
Dessert,757,63.61,38.0,91.64,92.0
Port/Sherry,358,131.53,50.0,91.62,92.0
Fortified,182,149.10,48.0,91.59,92.0
Port,92,136.14,43.0,92.04,92.0
Orange,78,30.36,25.0,90.06,90.0


## 11. Mean and median (price, rating) by features - age_at_review

In [5]:
# Integer bottle age, trimmed to a sensible 0-25y window (older tails are sparse).
age = (
    df[df["age_at_review"].between(0, 25)]
    .assign(age=lambda d: d["age_at_review"].astype(int))
)
by_age = agg_pr(age, "age", min_n=30).sort_index()
display(by_age)

fig = px.line(
    by_age.reset_index(), x="age", y=["retail_mean", "retail_median"],
    markers=True, title="Retail by bottle age at review",
    labels={"value": "Retail (USD)", "age": "Age at review (years)", "variable": "stat"},
)
fig.show()

fig = px.line(
    by_age.reset_index(), x="age", y=["rating_mean", "rating_median"],
    markers=True, title="Rating by bottle age at review",
    labels={"value": "Rating", "age": "Age at review (years)", "variable": "stat"},
)
fig.show()

,n,retail_mean,retail_median,rating_mean,rating_median
age,,,,,
0,269,18.03,16.0,87.90,88.0
1,27338,24.21,21.0,89.13,89.0
2,39431,38.49,30.0,90.35,90.0
3,30041,52.95,41.0,90.92,91.0
4,14071,57.88,47.0,90.92,91.0
5,6203,61.83,50.0,91.18,91.0
6,2552,71.56,55.0,91.47,92.0
7,1135,69.72,55.0,91.54,92.0
8,674,90.84,60.0,91.98,92.0


## 12. Mean and median (price, rating) by features - keywords (robust)

In [8]:
# Keyword densities are continuous; summarise each concept by comparing wines
# that *mention* it (density > 0) vs those that don't, on price and rating.
kr = pd.read_parquet(KW_ROBUST_PATH)
density_cols = [c for c in kr.columns if c != "wine_id" and not c.endswith("_count")]

m = (
    df[["wine_id", "retail", "rating"]]
    .merge(kr[["wine_id"] + density_cols], on="wine_id")
    .dropna(subset=["retail", "rating"])
)

rows = []
for c in density_cols:
    present = m[c] > 0
    if present.sum() < 50:  # skip near-empty concepts
        continue
    p, a = m[present], m[~present]
    rows.append({
        "concept": c.replace("kw_", ""),
        "pct_wines": round(present.mean() * 100, 1),
        "retail_med_present": round(p["retail"].median(), 1),
        "retail_med_absent": round(a["retail"].median(), 1),
        "rating_mean_present": round(p["rating"].mean(), 2),
        "rating_mean_absent": round(a["rating"].mean(), 2),
    })

kw_stats = pd.DataFrame(rows)
kw_stats["retail_lift"] = (kw_stats["retail_med_present"] - kw_stats["retail_med_absent"]).round(1)
kw_stats["rating_lift"] = (kw_stats["rating_mean_present"] - kw_stats["rating_mean_absent"]).round(2)
kw_stats = kw_stats.sort_values("pct_wines", ascending=False).reset_index(drop=True)

print(f"{len(kw_stats)} keyword concepts (>=50 mentions), sorted by prevalence:")
with pd.option_context("display.max_rows", None):
    display(kw_stats)

48 keyword concepts (>=50 mentions), sorted by prevalence:


,concept,pct_wines,retail_med_present,retail_med_absent,rating_mean_present,rating_mean_absent,retail_lift,rating_lift
0,tannic,36.6,38.0,30.0,90.77,90.10,8.0,0.67
1,acidic,32.9,30.0,34.0,90.34,90.35,-4.0,-0.01
2,citrus,29.8,29.0,35.0,90.40,90.32,-6.0,0.08
3,cherry,25.7,40.0,30.0,90.92,90.15,10.0,0.77
4,body,24.7,32.0,32.0,90.35,90.35,0.0,0.00
5,spicy,21.2,37.0,30.0,90.70,90.25,7.0,0.45
6,herbs,19.4,36.0,30.0,90.80,90.24,6.0,0.56
7,tart,16.6,30.0,32.0,90.33,90.35,-2.0,-0.02
8,earth,15.8,36.0,31.0,90.98,90.23,5.0,0.75
9,flowers,15.7,35.0,32.0,91.11,90.21,3.0,0.90


In [7]:
# Visualise each concept's price vs rating lift; size = how common the cue is.
fig = px.scatter(
    kw_stats, x="rating_lift", y="retail_lift", size="pct_wines", text="concept",
    title="Keyword cue lift: rating vs retail (present minus absent)",
    labels={"rating_lift": "Rating lift (pts)", "retail_lift": "Retail lift (USD, median)"},
)
fig.update_traces(textposition="top center")
fig.add_hline(y=0, line_dash="dot", line_color="grey")
fig.add_vline(x=0, line_dash="dot", line_color="grey")
fig.update_layout(height=620)
fig.show()